# Layer 5: Cross-Exchange Correlation
X1–X3: venue lag CDF, cumulative capacity, cascade throughput.

In [ ]:
import os, numpy as np, pandas as pd

DATA_DIR    = os.environ.get("DATA_DIR",    "/data/parquet")
REPORTS_DIR = os.environ.get("REPORTS_DIR", "/data/reports")
os.makedirs(REPORTS_DIR, exist_ok=True)

from mda.loader import load_trades
from mda.layer1.rates import compute_rate_timeseries, compute_rate_percentiles
from mda.layer4.update_rates import compute_update_rate_by_depth
from mda.layer5.cross_exchange import (
    compute_cross_venue_lag, compute_cumulative_capacity, detect_cascade_events,
)
from mda.plots.charts import (
    plot_cross_venue_lag_cdf, plot_cumulative_capacity, plot_cascade_throughput,
    save_figure,
)
print("imports OK")

In [ ]:
df = load_trades(DATA_DIR)
print(f"Loaded {len(df):,} trades")
rate_ts   = compute_rate_timeseries(df)
rate_pcts = compute_rate_percentiles(rate_ts)
print("Rate p99 per exchange:")
print(rate_pcts[["exchange","p99","p99_9"]].to_string(index=False))

In [ ]:
lags = compute_cross_venue_lag(df, symbol_filter="BTC", sample_frac=0.1)
print(f"Cross-venue lag pairs: {lags.groupby(['exchange_a','exchange_b']).size().to_dict()}")

In [ ]:
fig = plot_cross_venue_lag_cdf(lags)
fig.show()
save_figure(fig, "X1_cross_venue_lag_cdf", REPORTS_DIR)

In [ ]:
import gc
from mda.loader import load_orderbook

# Load only columns needed for update rate (exchange, level, received_at).
# Full OB load is 24 GB for Binance alone; minimal cols = ~1 GB.
EXCHANGES_OB = df["exchange"].unique().tolist()
ob_rate_parts = []
for exch in EXCHANGES_OB:
    ob_exch = load_orderbook(DATA_DIR, exchanges=[exch],
                             columns=["exchange", "level", "received_at"],
                             add_ts_cols=False)
    ob_exch["receive_ts_dt"] = __import__("pandas").to_datetime(
        ob_exch["received_at"], utc=True, format="mixed")
    ob_rate_parts.append(compute_update_rate_by_depth(ob_exch))
    del ob_exch
    gc.collect()

ob_update_rate = __import__("pandas").concat(ob_rate_parts, ignore_index=True)
del ob_rate_parts
gc.collect()

capacity = compute_cumulative_capacity(rate_pcts, ob_update_rate)
print("Cumulative capacity:")
print(capacity.to_string(index=False))

In [ ]:
fig = plot_cumulative_capacity(capacity)
fig.show()
save_figure(fig, "X2_cumulative_capacity", REPORTS_DIR)

In [ ]:
cascade_windows = detect_cascade_events(df, symbol_filter="BTC", pct_threshold=0.02)
print(f"Cascade events detected: {len(cascade_windows)}")
fig = plot_cascade_throughput(rate_ts, cascade_windows)
fig.show()
save_figure(fig, "X3_cascade_throughput", REPORTS_DIR)